
# Topic 3 — Continued Pretraining / Domain Adaptation

**Core question:** *How do I teach a model domain language and knowledge patterns?*

**Running example continues**, but the task shape changes completely. Instead
of (problem → response) pairs, we train on **raw domain text** with plain
next-token prediction — no labels, no chat template, no "correct answer."

```mermaid
flowchart LR
    subgraph SFT["Topic 1: SFT"]
    S1["Question"] --> S2["Answer"]
    end
    subgraph CPT["Topic 3: Continued pretraining"]
    C1["Raw domain text"] --> C2["Next-token prediction<br/>(predict every next word)"]
    end
```

**The key distinction to internalize:**
- **SFT** (Topic 1): `Question → Answer` — teaches the model a specific
  input/output *behavior*.
- **Continued pretraining** (this notebook): `Raw domain text → next-token
  prediction` — teaches the model domain *language and knowledge patterns*
  by simply having it keep practicing prediction on text from that domain.

Our tiny synthetic domain corpus covers PostgreSQL internals, Java,
distributed systems, and ServiceNow/Glide-platform docs — a stand-in for
"5B tokens of technical corpus," scaled down ~1,000,000x so it runs on
Colab's free tier.

### Can you answer this?
1. When would you pick continued pretraining over SFT?
2. Why is continued pretraining usually run with a *smaller* learning rate than SFT?
3. What's actually different about the training objective compared to SFT?


In [1]:

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box

console = Console()
def banner(title, subtitle="", style="bold white on blue"):
    text = title if not subtitle else f"{title}\n[dim]{subtitle}[/dim]"
    console.print(Panel(text, style=style, expand=False, box=box.DOUBLE))
def section(title):
    console.print(f"\n[bold cyan]── {title} " + "─" * max(0, 60 - len(title)) + "[/bold cyan]")
def kv_table(title, rows, caption=""):
    table = Table(title=title, box=box.SIMPLE_HEAVY, caption=caption)
    table.add_column("Key", style="bold")
    table.add_column("Value")
    for k, v in rows.items():
        table.add_row(str(k), str(v))
    console.print(table)

banner("Topic 3: Continued Pretraining / Domain Adaptation",
       "Raw technical corpus -> next-token prediction")


╔════════════════════════════════════════════════════╗
║ Topic 3: Continued Pretraining / Domain Adaptation ║
║ Raw technical corpus -> next-token prediction      ║
╚════════════════════════════════════════════════════╝


## Concept primer

**Domain adaptation** is the broader goal: making a general-purpose model
more fluent and knowledgeable in a specific domain (legal, medical,
PostgreSQL internals, your company's codebase, etc.) by continuing training
on domain text.

**Corpus construction**: unlike SFT's curated (input, output) pairs,
continued pretraining just needs a large pile of representative domain
text. Quality still matters -- garbage text in, garbage patterns out -- but
you don't need labels or a specific task shape.

**Data mixtures**: in practice you rarely train on *only* new domain text --
you mix in some of the original pretraining-style general text too, to
avoid regressing general ability while gaining domain fluency. The mixture
ratio (e.g. 80% domain / 20% general) is itself a tuned hyperparameter.

**Learning rate selection**: continued pretraining typically uses a
*smaller* learning rate than SFT (e.g. 1e-5 to 5e-5 vs. SFT's 1e-4 to 2e-4).
The model already "knows" general language; we want to nudge it toward
domain patterns without the aggressive updates that would overwrite that
general knowledge.

**Forgetting**: the same catastrophic forgetting risk from Topic 1 applies
here too, and is often *more* subtle -- because there's no explicit
right/wrong label, it's easy to over-train on a narrow domain and not notice
until you test general capability separately.

**Domain-specific vocabulary**: domain text is full of terms (`MVCC`,
`autovacuum`, `sys_attachment_doc`, `CompletableFuture`) that a general
tokenizer may not have as single tokens -- we'll check this directly below.

**Continued pretraining vs. SFT** (the headline distinction): SFT shapes
*behavior* for a specific task; continued pretraining shapes *fluency and
recall* for a domain, independent of any particular task format. You often
do continued pretraining *first*, then SFT *on top of* the domain-adapted
model, to get both benefits.


In [2]:

import torch
section("Environment check")
HAS_CUDA = torch.cuda.is_available()
kv_table("Runtime", {"CUDA available": HAS_CUDA, "PyTorch": torch.__version__})
if not HAS_CUDA:
    console.print(Panel(
        "[bold yellow]No CUDA GPU detected.[/bold yellow] Corpus prep, tokenization, and "
        "chunking below are all real CPU-safe work. Actual continued-pretraining and "
        "before/after domain-eval generation require a GPU -- run on Colab (T4) for those.",
        title="GPU NOT AVAILABLE HERE", style="yellow", box=box.HEAVY,
    ))


── Environment check ───────────────────────────────────────────

          Runtime          
                           
  Key              Value   
 ━━━━━━━━━━━━━━━━━━━━━━━━━ 
  CUDA available   False   
  PyTorch          2.14.1

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ GPU NOT AVAILABLE HERE ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ No CUDA GPU detected. Corpus prep, tokenization, and chunking below are all real CPU-safe work. Actual          ┃
┃ continued-pretraining and before/after domain-eval generation require a GPU -- run on Colab (T4) for those.     ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

In [3]:

import sys
IN_COLAB = "google.colab" in sys.modules
packages = ["unsloth", "transformers", "trl", "peft", "bitsandbytes", "torch", "datasets", "rich"]
if IN_COLAB:
    get_ipython().system(f"pip install -q {' '.join(packages)}")
    console.print("[green]Installed packages for Colab.[/green]")
else:
    console.print("[dim]Not running in Colab -- assuming the project virtualenv already has these.[/dim]")


Not running in Colab -- assuming the project virtualenv already has these.

In [4]:
# --- Load the domain corpus --------------------------------------------------
import json
import random
from pathlib import Path

section("Loading the domain corpus")

vsc_ipynb_file = globals().get("__vsc_ipynb_file__")
search_roots = [Path.cwd()]
if vsc_ipynb_file:
    search_roots.insert(0, Path(vsc_ipynb_file).resolve().parent)
search_roots += [root.parent for root in list(search_roots)]

RELATIVE_CANDIDATES = ["data/domain_corpus.txt", "03_continued_pretraining/data/domain_corpus.txt"]
CORPUS_PATH = None
for root in search_roots:
    for rel in RELATIVE_CANDIDATES:
        candidate = root / rel
        if candidate.is_file():
            CORPUS_PATH = candidate
            break
    if CORPUS_PATH:
        break

if CORPUS_PATH is None:
    # Self-sufficient fallback: regenerate the exact same deterministic corpus
    # (and its matching domain eval set) in-memory instead of requiring a
    # separate file upload (e.g. on Colab).
    console.print("[dim]domain_corpus.txt not found nearby -- generating it in-memory "
                  "instead of requiring a separate file upload.[/dim]")

    PG_TEMPLATES = [
        "PostgreSQL uses {mechanism} to implement {feature}. {detail}",
        "In {feature}, the {component} is responsible for {action}. {detail}",
        "A common cause of {problem} in PostgreSQL is {cause}. The usual fix is {fix}.",
    ]
    PG_SLOTS = {
        "mechanism": ["multi-version concurrency control (MVCC)", "write-ahead logging (WAL)",
                      "the buffer cache", "autovacuum", "the planner's cost model", "table partitioning"],
        "feature": ["transaction isolation", "crash recovery", "index-only scans",
                    "vacuum and freezing", "replication", "query planning"],
        "component": ["background writer", "WAL writer", "autovacuum launcher",
                      "checkpointer", "query planner", "buffer manager"],
        "action": ["flushing dirty pages to disk", "reclaiming dead tuples",
                   "writing WAL records before data pages", "estimating row counts",
                   "choosing between a sequential scan and an index scan"],
        "detail": [
            "This avoids readers blocking writers and writers blocking readers.",
            "This ensures durability even if the server crashes mid-transaction.",
            "This keeps table and index bloat under control over time.",
            "This trades some write amplification for faster crash recovery.",
            "This relies on accurate table statistics gathered by ANALYZE.",
        ],
        "problem": ["table bloat", "replication lag", "long-running transaction blocking vacuum",
                    "query plan regressions", "connection pool exhaustion"],
        "cause": ["autovacuum not keeping up with the update/delete rate",
                  "a long-running transaction holding back the vacuum horizon",
                  "stale table statistics after a bulk load",
                  "too many idle connections holding resources"],
        "fix": ["tuning autovacuum thresholds per table", "running a manual VACUUM (or VACUUM FULL)",
                "running ANALYZE after large data changes", "adding a connection pooler like PgBouncer"],
    }

    JAVA_TEMPLATES = [
        "In Java, {concept} is used to {purpose}. {detail}",
        "The {component} in the JVM is responsible for {action}. {detail}",
        "A common mistake with {concept} is {mistake}, which leads to {consequence}.",
    ]
    JAVA_SLOTS = {
        "concept": ["garbage collection", "the ConcurrentHashMap", "try-with-resources",
                    "generics", "the volatile keyword", "CompletableFuture"],
        "purpose": ["automatically reclaim memory from unreachable objects",
                    "allow safe concurrent reads and writes without a global lock",
                    "guarantee a resource is closed even if an exception is thrown",
                    "catch type errors at compile time instead of at runtime",
                    "ensure visibility of a variable's latest value across threads",
                    "compose asynchronous operations without blocking a thread"],
        "detail": [
            "Different garbage collectors (G1, ZGC, Shenandoah) trade pause time against throughput.",
            "This scales much better under contention than a synchronized HashMap.",
            "This is equivalent to an implicit finally block closing the resource.",
            "Type erasure means generic type information isn't available at runtime.",
            "It does not make compound operations atomic on its own.",
            "Exceptions inside the chain propagate through the returned future.",
        ],
        "component": ["garbage collector", "JIT compiler", "classloader", "thread scheduler"],
        "action": ["reclaiming memory from objects with no remaining references",
                   "compiling hot methods to native code at runtime",
                   "loading class bytecode into the running JVM",
                   "deciding which runnable thread executes next"],
        "mistake": ["assuming += on a shared counter is atomic", "forgetting to override hashCode when overriding equals",
                    "catching Exception too broadly", "blocking inside a reactive/async callback"],
        "consequence": ["lost updates under concurrent access", "broken behavior in hash-based collections",
                        "swallowing bugs that should have failed loudly", "thread pool starvation"],
    }

    DIST_TEMPLATES = [
        "In distributed systems, {concept} addresses the problem of {problem}. {detail}",
        "{algorithm} achieves {guarantee} by {mechanism}.",
        "A key trade-off in distributed systems is between {tradeoff_a} and {tradeoff_b}. {detail}",
    ]
    DIST_SLOTS = {
        "concept": ["consensus", "leader election", "vector clocks", "quorum reads/writes",
                    "idempotent request handling", "backpressure"],
        "problem": ["multiple nodes agreeing on a single value despite failures",
                    "ensuring exactly one node coordinates a given task",
                    "ordering events across nodes without a shared clock",
                    "guaranteeing a majority of replicas have seen a write"],
        "detail": [
            "Protocols like Raft and Paxos solve this with majority voting over terms/rounds.",
            "This typically uses lease-based mechanisms with periodic renewal.",
            "This provides a partial order, not a total order, across events.",
            "Reading and writing to a majority of replicas tolerates minority failures.",
        ],
        "algorithm": ["Raft", "Paxos", "Two-Phase Commit", "Gossip protocols"],
        "guarantee": ["strong consistency across replicas", "atomic commitment across participants",
                      "eventual convergence of cluster state"],
        "mechanism": ["electing a leader and replicating a log with majority acknowledgment",
                      "a coordinator collecting votes before committing or aborting",
                      "nodes periodically exchanging state with random peers"],
        "tradeoff_a": ["consistency", "latency", "availability during a partition"],
        "tradeoff_b": ["availability", "throughput", "strict ordering guarantees"],
    }

    SN_TEMPLATES = [
        "On the ServiceNow platform, {feature} is implemented by {component}. {detail}",
        "When {scenario}, the platform relies on {mechanism} to {outcome}.",
        "A common operational concern with {feature} is {problem}, mitigated by {fix}.",
    ]
    SN_SLOTS = {
        "feature": ["table archiving", "attachment storage partitioning", "ACL enforcement",
                    "business rule execution order", "the Unreferenced Record Cleaner job"],
        "component": ["a scheduled background job that moves qualifying records to archive tables",
                      "a provider abstraction that routes attachments to the active storage partition",
                      "the security manager evaluating role and condition checks per operation",
                      "the business rule engine ordering rules by order field within each trigger phase"],
        "detail": [
            "This keeps the live table smaller while still letting archived data be queried via ar_ tables.",
            "This allows storage to scale horizontally as attachment volume grows.",
            "This runs on every record access, not just on save.",
            "Rules in the same order value execute in an implementation-defined but stable sequence.",
        ],
        "scenario": ["a record is dot-walked across tables on an archived form",
                     "a large table accumulates orphaned child records",
                     "an instance needs to add a new storage partition without downtime"],
        "mechanism": ["the ArchiveResolver fallback path", "a cleanup job that walks reference fields",
                      "a monitor job that triggers partition creation above a size threshold"],
        "outcome": ["resolve the correct archived parent record", "safely delete unreferenced rows",
                    "route new uploads to the newly created partition"],
        "problem": ["forgetting to account for archived data in integration tests",
                    "job contention when many large tables archive at the same time",
                    "stale cached state after a schema change"],
        "fix": ["adding ar_ table coverage explicitly to IT suites",
                "staggering archive job schedules across tables",
                "invalidating the relevant caches on schema change"],
    }

    def _render(templates, slots, rng, n):
        paragraphs = []
        for _ in range(n):
            template = rng.choice(templates)
            filled = template
            for key, options in slots.items():
                placeholder = "{" + key + "}"
                while placeholder in filled:
                    filled = filled.replace(placeholder, rng.choice(options), 1)
            paragraphs.append(filled)
        return paragraphs

    def generate_domain_corpus(seed=42, per_domain=25):
        rng = random.Random(seed)
        domains = {
            "postgresql_internals": _render(PG_TEMPLATES, PG_SLOTS, rng, per_domain),
            "java": _render(JAVA_TEMPLATES, JAVA_SLOTS, rng, per_domain),
            "distributed_systems": _render(DIST_TEMPLATES, DIST_SLOTS, rng, per_domain),
            "servicenow_platform": _render(SN_TEMPLATES, SN_SLOTS, rng, per_domain),
        }
        all_paragraphs = [p for paras in domains.values() for p in paras]
        rng.shuffle(all_paragraphs)
        return all_paragraphs

    DOMAIN_EVAL_SET = [
        {"type": "terminology", "prompt": "What does MVCC stand for and why does PostgreSQL use it?"},
        {"type": "terminology", "prompt": "What is the purpose of Write-Ahead Logging (WAL)?"},
        {"type": "qa", "prompt": "Why can a long-running transaction block autovacuum in PostgreSQL?"},
        {"type": "qa", "prompt": "What problem does Raft's leader election solve?"},
        {"type": "summarization", "prompt": "Summarize in one sentence why ServiceNow uses table archiving."},
        {"type": "summarization", "prompt": "Summarize the trade-off between consistency and availability in distributed systems."},
        {"type": "reasoning", "prompt": "If a ServiceNow instance archives a table but doesn't update integration tests, what's likely to break?"},
        {"type": "reasoning", "prompt": "Why might using CompletableFuture incorrectly cause thread pool starvation?"},
    ]

    all_paragraphs = generate_domain_corpus()
    corpus_text = "\n\n".join(all_paragraphs)
    CORPUS_PATH = Path.cwd() / "data" / "domain_corpus.txt"
    CORPUS_PATH.parent.mkdir(parents=True, exist_ok=True)
    CORPUS_PATH.write_text(corpus_text)
    with (CORPUS_PATH.parent / "domain_eval.json").open("w") as f:
        json.dump(DOMAIN_EVAL_SET, f, indent=2)
else:
    corpus_text = CORPUS_PATH.read_text()

paragraphs = [p for p in corpus_text.split("\n\n") if p.strip()]
word_count = len(corpus_text.split())
console.print(f"Loaded [bold]{len(paragraphs)}[/bold] paragraphs, [bold]{word_count}[/bold] words from {CORPUS_PATH}")
console.print(Panel(paragraphs[0], title="Sample paragraph", box=box.ROUNDED))


── Loading the domain corpus ───────────────────────────────────

Loaded 100 paragraphs, 2431 words from 
/Users/bhanupraka.gopularam/aiprojects/llm-fine-tuning/03_continued_pretraining/data/domain_corpus.txt

╭─────────────────────────────────────────────── Sample paragraph ────────────────────────────────────────────────╮
│ The JIT compiler in the JVM is responsible for deciding which runnable thread executes next. This scales much   │
│ better under contention than a synchronized HashMap.                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [5]:

# --- Tokenize + chunk the corpus for next-token-prediction training ---------
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

section("Tokenizing the corpus")
all_token_ids = tokenizer(corpus_text)["input_ids"]
console.print(f"Full corpus tokenizes to [bold]{len(all_token_ids)}[/bold] tokens "
              f"({len(all_token_ids)/word_count:.2f} tokens/word)")

BLOCK_SIZE = 128
blocks = [all_token_ids[i:i + BLOCK_SIZE] for i in range(0, len(all_token_ids), BLOCK_SIZE)]
blocks = [b for b in blocks if len(b) == BLOCK_SIZE]  # drop the final partial block
console.print(f"Chunked into [bold]{len(blocks)}[/bold] fixed-length blocks of {BLOCK_SIZE} tokens each "
              "-- this is the actual training unit for continued pretraining "
              "(no prompt/response split, every token in the block is both input and target).")


── Tokenizing the corpus ───────────────────────────────────────

Full corpus tokenizes to 2972 tokens (1.22 tokens/word)

Chunked into 23 fixed-length blocks of 128 tokens each -- this is the actual training unit for continued 
pretraining (no prompt/response split, every token in the block is both input and target).

In [6]:

# --- Domain vocabulary tokenization check -----------------------------------
section("Domain vocabulary: how many tokens does Qwen3's tokenizer need per term?")

domain_terms = ["MVCC", "autovacuum", "WAL", "tablespace", "ConcurrentHashMap",
                 "CompletableFuture", "Raft", "Paxos", "sys_attachment_doc",
                 "ServiceNow", "PgBouncer", "idempotent"]

table = Table(title="Domain term -> token breakdown (real tokenizer output)", box=box.SIMPLE_HEAVY)
table.add_column("Term")
table.add_column("# tokens", justify="right")
table.add_column("Tokens")
for term in domain_terms:
    pieces = tokenizer.tokenize(term)
    table.add_row(term, str(len(pieces)), str(pieces))
console.print(table)
console.print(
    "\n[dim]Terms that split into many sub-word pieces are ones the tokenizer wasn't built "
    "around -- the model has to learn to 'reassemble' their meaning from fragments, which is "
    "part of why continued pretraining on domain text helps: it gives the model many more "
    "examples of these fragment sequences appearing together in context.[/dim]"
)


── Domain vocabulary: how many tokens does Qwen3's tokenizer need per term? 

      Domain term -> token breakdown (real tokenizer output)      
                                                                  
  Term                 # tokens   Tokens                          
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  MVCC                        2   ['MV', 'CC']                    
  autovacuum                  4   ['aut', 'ov', 'ac', 'uum']      
  WAL                         2   ['W', 'AL']                     
  tablespace                  2   ['table', 'space']              
  ConcurrentHashMap           3   ['Con', 'current', 'HashMap']   
  CompletableFuture           2   ['Comple', 'tableFuture']       
  Raft                        2   ['Ra', 'ft']                    
  Paxos                       3   ['P', 'ax', 'os']               
  sys_attachment_doc          3   ['sys', '_attachment', '_doc']  
  ServiceNow                  2   ['Service', 'Now']              
  PgBouncer                   3   ['Pg', 'B', 'ouncer']           
  idempotent                  3   ['id', 'emp', 'otent']

Terms that split into many sub-word pieces are ones the tokenizer wasn't built around -- the model has to learn to 
'reassemble' their meaning from fragments, which is part of why continued pretraining on domain text helps: it 
gives the model many more examples of these fragment sequences appearing together in context.


## Continued pretraining setup

Unlike `SFTTrainer`, continued pretraining is a plain causal-language-model
training loop: every token in a block is used as both input and label
(shifted by one position), via `DataCollatorForLanguageModeling`. There's no
prompt/response masking because there's no prompt/response structure.


In [7]:

# --- Build the LM dataset + trainer config (CPU-safe) ------------------------
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

lm_dataset = Dataset.from_dict({"input_ids": blocks})
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

kv_table("Continued pretraining config (this run)", {
    "block_size": BLOCK_SIZE,
    "num_blocks": len(blocks),
    "learning_rate": "2e-5  (vs. 2e-4 used for SFT in Topic 1 -- 10x smaller)",
    "num_train_epochs": 3,
    "objective": "causal language modeling (predict every next token)",
})


                   Continued pretraining config (this run)                    
                                                                              
  Key                Value                                                    
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  block_size         128                                                      
  num_blocks         23                                                       
  learning_rate      2e-5  (vs. 2e-4 used for SFT in Topic 1 -- 10x smaller)  
  num_train_epochs   3                                                        
  objective          causal language modeling (predict every next token)

In [8]:

# --- Run continued pretraining (GPU-only) + before/after domain eval --------
import json

eval_path = CORPUS_PATH.parent / "domain_eval.json"
with open(eval_path) as f:
    domain_eval = json.load(f)

if HAS_CUDA:
    from unsloth import FastLanguageModel
    from transformers import TrainingArguments, Trainer

    model, tok = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=512)

    def generate(model, tok, prompt, max_new_tokens=120):
        inputs = tok(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        return tok.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()

    FastLanguageModel.for_inference(model)
    section("Baseline (before continued pretraining) answers")
    baseline = {}
    for item in domain_eval[:3]:
        out = generate(model, tok, item["prompt"])
        baseline[item["prompt"]] = out
        console.print(Panel(f"[bold]Q:[/bold] {item['prompt']}\n\n[bold]Base model:[/bold]\n{out}", box=box.ROUNDED))

    FastLanguageModel.for_training(model)
    args = TrainingArguments(output_dir="outputs/continued_pretraining", num_train_epochs=3,
                               per_device_train_batch_size=4, learning_rate=2e-5,
                               logging_steps=10, save_strategy="no", report_to=[])
    trainer = Trainer(model=model, args=args, train_dataset=lm_dataset, data_collator=data_collator)
    section("Running continued pretraining")
    result = trainer.train()
    kv_table("Training result", {"final loss": result.training_loss})

    FastLanguageModel.for_inference(model)
    section("After continued pretraining -- same questions")
    for item in domain_eval[:3]:
        out = generate(model, tok, item["prompt"])
        table = Table(title=item["prompt"], box=box.SIMPLE_HEAVY, show_lines=True)
        table.add_column("Before", style="red", width=55)
        table.add_column("After", style="green", width=55)
        table.add_row(baseline[item["prompt"]], out)
        console.print(table)
else:
    console.print(Panel(
        "[yellow]SKIPPED[/yellow] -- continued pretraining and domain-eval generation require a CUDA GPU.\n"
        "Expected on Colab (T4): baseline answers that are generic/non-specific about these domain\n"
        "terms, a training loss curve over the small corpus, then after-training answers that more\n"
        "often reference the specific terminology/relationships seen in the synthetic corpus\n"
        "(though with a corpus this tiny, don't expect deep new knowledge -- the point is observing\n"
        "the *mechanism*, not producing a genuinely domain-expert model).",
        title="Continued pretraining + domain eval (GPU-only)", style="yellow",
    ))


╭──────────────────────────────── Continued pretraining + domain eval (GPU-only) ─────────────────────────────────╮
│ SKIPPED -- continued pretraining and domain-eval generation require a CUDA GPU.                                 │
│ Expected on Colab (T4): baseline answers that are generic/non-specific about these domain                       │
│ terms, a training loss curve over the small corpus, then after-training answers that more                       │
│ often reference the specific terminology/relationships seen in the synthetic corpus                             │
│ (though with a corpus this tiny, don't expect deep new knowledge -- the point is observing                      │
│ the *mechanism*, not producing a genuinely domain-expert model).                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Can you answer this? (Topic 3 training checkpoint)

**When would you pick continued pretraining over SFT?**
<details><summary>Answer</summary>
When the gap is *knowledge/fluency*, not *behavior*. If the model doesn't
know your domain's vocabulary, facts, or typical phrasing at all, SFT on a
few hundred Q&A pairs won't fix that -- there isn't enough raw exposure to
the domain's language for the model to generalize from. Continued
pretraining on a large volume of domain text builds that foundation; SFT
then shapes how the (now domain-fluent) model responds to specific tasks.
In practice: continued pretraining first (if you have the domain corpus),
SFT second (for the actual task behavior).
</details>

**Why is continued pretraining usually run with a smaller learning rate than SFT?**
<details><summary>Answer</summary>
Continued pretraining text is unstructured and much larger in volume than a
curated SFT set -- there's no "right answer" signal shaping each update, just
broad exposure. A smaller learning rate makes the nudge toward domain
patterns gentler, reducing the risk of catastrophically overwriting the
model's existing general-purpose knowledge while it processes a large
quantity of new text.
</details>

**What's actually different about the training objective compared to SFT?**
<details><summary>Answer</summary>
Mechanically, both use cross-entropy next-token loss. The difference is
*where* the loss is computed: SFT masks the loss to only the response
tokens (ignoring the prompt); continued pretraining computes loss over
*every* token in the block, since there's no prompt/response distinction in
raw text.
</details>


In [9]:

section("Scale discussion: continued pretraining at ~100 vs ~10k users")
table = Table(title="Domain adaptation at scale", box=box.SIMPLE_HEAVY, show_lines=True)
table.add_column("Concern")
table.add_column("~100 users", style="cyan")
table.add_column("~10k users", style="magenta")
rows = [
    ("Corpus size needed", "A few hundred MB of domain text can meaningfully help",
                           "Real gains typically need GBs-to-TBs of curated domain text -- corpus curation becomes a dedicated pipeline, not a one-off script"),
    ("Training cost", "Hours on a single GPU for a small model",
                      "Many GPU-days/weeks; often done on a cluster, not a single Colab-style session"),
    ("Data mixture tuning", "Can eyeball a reasonable domain/general split",
                            "Needs systematic ablations to find the mixture ratio that avoids regressing general capability"),
    ("Refresh cadence", "Ad hoc, when domain material changes meaningfully",
                        "Periodic (e.g. quarterly) as the domain's documentation/codebase evolves, often paired with automated regression evals"),
]
for c, s, b in rows:
    table.add_row(c, s, b)
console.print(table)


── Scale discussion: continued pretraining at ~100 vs ~10k users 

                                            Domain adaptation at scale                                             
                                                                                                                   
  Concern               ~100 users                                    ~10k users                                   
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Corpus size needed    A few hundred MB of domain text can           Real gains typically need GBs-to-TBs of      
                        meaningfully help                             curated domain text -- corpus curation       
                                                                      becomes a dedicated pipeline, not a one-off  
                                                                      script                                       
                                                                                                                   
  Training cost         Hours on a single GPU for a small model       Many GPU-days/weeks; often done on a         
                                                                      cluster, not a single Colab-style session    
                                                                                                                   
  Data mixture tuning   Can eyeball a reasonable domain/general       Needs systematic ablations to find the       
                        split                                         mixture ratio that avoids regressing         
                                                                      general capability                           
                                                                                                                   
  Refresh cadence       Ad hoc, when domain material changes          Periodic (e.g. quarterly) as the domain's    
                        meaningfully                                  documentation/codebase evolves, often        
                                                                      paired with automated regression evals      

In [10]:

section("Summary")
console.print(Panel(
    "[bold]What we did:[/bold]\n"
    "  1. Generated a tiny synthetic multi-domain corpus (PostgreSQL, Java, distributed systems, ServiceNow)\n"
    "  2. Tokenized and chunked it into fixed-length blocks for next-token-prediction training\n"
    "  3. Inspected how the tokenizer handles real domain vocabulary\n"
    "  4. Configured a continued-pretraining run (CLM objective, small LR)\n"
    "  5. (On GPU) ran it and compared before/after answers on a domain eval set\n\n"
    "[bold]Next: Topic 4 -- Preference Optimization + RL[/bold]\n"
    "SFT -> DPO -> GRPO on a simple math reasoning task.",
    title="Topic 3 complete", style="bold green", box=box.DOUBLE,
))


── Summary ─────────────────────────────────────────────────────

╔═══════════════════════════════════════════════ Topic 3 complete ════════════════════════════════════════════════╗
║ What we did:                                                                                                    ║
║   1. Generated a tiny synthetic multi-domain corpus (PostgreSQL, Java, distributed systems, ServiceNow)         ║
║   2. Tokenized and chunked it into fixed-length blocks for next-token-prediction training                       ║
║   3. Inspected how the tokenizer handles real domain vocabulary                                                 ║
║   4. Configured a continued-pretraining run (CLM objective, small LR)                                           ║
║   5. (On GPU) ran it and compared before/after answers on a domain eval set                                     ║
║                                                                                                                 ║
║ Next: Topic 4 -- Preference Optimization + RL                                                                   ║
║ SFT -> DPO -> GRPO on a simple math reasoning task.                                                             ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝